# Pay for Secure Data (x402)

This notebook walks through the public-safe parts of the sample: environment loading, runtime imports, t54 x402-secure trust scoring, plugin-compatible HTTP 402 response shape, target x402 guardrail behavior, and optional AWS/x402 prerequisite gates. It does not execute a live payment unless you wire your own funded test endpoint outside this notebook.

## What This Sample Shows

Neighboring `pay-for-data` and `pay-for-api-agent` samples use `AgentCorePaymentsPlugin` to make paid x402 HTTP calls look like ordinary tool calls. This sample uses the same plugin-native pattern for both t54 x402-secure and a registered target x402 service, while keeping the guardrail deterministic: `check_x402_endpoint_trust` stores the t54 score in request-scoped state, and `call_trusted_x402_service` refuses to call the target endpoint unless that exact service endpoint has passed.

## Flow

1. Copy `.env.example` to `.env`.
2. Run local unit tests without AWS credentials.
3. Optionally run `setup/setup_roles.sh` and `setup/setup_manager.sh` after reviewing IAM scope.
4. Invoke the runtime with a prompt and `payment_context`.
5. The agent calls `check_x402_endpoint_trust`; if t54 x402-secure returns HTTP 402, `AgentCorePaymentsPlugin` generates the x402 proof and retries the tool.
6. The successful trust score is stored in request-scoped state for the exact registered endpoint URL.
7. The agent calls `call_trusted_x402_service`; code-level guardrails block low-quality, scam, missing, expired, or URL-mismatched trust state before the target endpoint is called.
8. If the guardrail passes, the registered target endpoint payment is also handled by `AgentCorePaymentsPlugin`.

## Prerequisites

- Python environment with the sample dependencies installed.
- A sample-root `.env` copied from `.env.example`.
- Local tests do not require AWS credentials.
- Live paid requests require a funded instrument and public x402 endpoints you control or are authorized to call.

In [ ]:
from pathlib import Path
import os
import sys

SAMPLE_ROOT = Path.cwd()
AGENT_ROOT = SAMPLE_ROOT / "agent"
sys.path.insert(0, str(AGENT_ROOT))

os.environ.setdefault("AWS_REGION", "us-west-2")
os.environ.setdefault("X402_TRUST_THRESHOLD", "50")
os.environ.setdefault("X402_TRUST_FAIL_CLOSED", "1")
print(f"Sample root: {SAMPLE_ROOT}")

In [ ]:
import agent
import main
import payments
import x402_secure
import x402_services

print("Runtime imports ok")
print("Registered x402 services:", ", ".join(x402_services.supported_service_ids()))
print("Heurist operations:", ", ".join(x402_services.supported_service_operations()[:5]), "...")
print("Agent tools: check_x402_endpoint_trust, call_trusted_x402_service")

## Local Mock: t54 x402-secure Guardrail

This mock demonstrates the deterministic guardrail without contacting AWS or moving funds. In the runtime, a real trust check uses `check_x402_endpoint_trust` to call `https://x402-secure-api.t54.ai/x402/tools/get_overall_score`; `AgentCorePaymentsPlugin` handles any HTTP 402 payment retry and the successful score is stored in request-scoped trust state for the exact registered target endpoint.

In [ ]:
class FakeTrustClient:
    def __init__(self, score):
        self.score = score
        self.calls = []

    def score_endpoint(self, url, *, headers=None):
        self.calls.append((url, headers))
        return self.score


class FakeServiceClient:
    def __init__(self):
        self.calls = []

    def call_operation(self, operation, payload, *, headers=None):
        self.calls.append((operation, payload, headers))
        return {"quotes": [{"symbol": payload["symbols"][0], "price": 188.22}]}


def demo_registry(service_client):
    return {
        "heurist_yahoo_finance": {
            "base_url": "https://target.example/x402/yahoo",
            "operations": {
                "quote_snapshot": {"required": {"symbols"}},
            },
            "client": service_client,
        }
    }


trusted_score = {"overall_score": 91, "risk_level": "low", "is_scam": False}
trusted_service = FakeServiceClient()
trusted_gateway = x402_services.TrustedX402ServiceGateway(
    services=demo_registry(trusted_service),
    trust_client=FakeTrustClient(trusted_score),
    trust_threshold=50,
)
with x402_services.use_request_trust_state():
    trusted_gateway.check_x402_endpoint_trust(
        service_id="heurist_yahoo_finance",
        headers={"X-PAYMENT": "trust-proof"},
    )
    trusted_result = trusted_gateway.call_trusted_x402_service(
        "heurist_yahoo_finance",
        "quote_snapshot",
        {"symbols": ["AAPL"]},
        headers={"PAYMENT-SIGNATURE": "data-proof"},
    )
trusted_result

## Local Mock: Block Low-Quality Endpoint

When x402-secure returns a score below the threshold, the target data endpoint is not called. This is the key difference from a plain x402 payment sample.

In [ ]:
low_score = {
    "overall_score": 23,
    "risk_level": "high",
    "is_scam": False,
    "scam_indicators": ["fresh wallet", "thin web presence"],
}
blocked_service = FakeServiceClient()
blocked_gateway = x402_services.TrustedX402ServiceGateway(
    services=demo_registry(blocked_service),
    trust_client=FakeTrustClient(low_score),
    trust_threshold=50,
)
with x402_services.use_request_trust_state():
    blocked_gateway.check_x402_endpoint_trust(service_id="heurist_yahoo_finance")
    blocked_result = blocked_gateway.call_trusted_x402_service(
        "heurist_yahoo_finance",
        "quote_snapshot",
        {"symbols": ["AAPL"]},
    )
assert blocked_result["status"] == "blocked"
assert blocked_service.calls == []
blocked_result

## Local Mock: Plugin-Compatible x402 Retry Shape

Both the t54 trust check and the target data call expose the same tool contract to `AgentCorePaymentsPlugin`: the first call can return an HTTP 402-shaped result, and the plugin retries the same tool with an x402 payment header. The tool itself does not call `ProcessPayment`.

In [ ]:
class FakeResponse:
    def __init__(self, status_code, payload, headers=None):
        self.status_code = status_code
        self._payload = payload
        self.headers = headers or {}
        self.text = str(payload)

    def json(self):
        return self._payload


class FakeSession:
    def __init__(self, responses):
        self.responses = list(responses)
        self.calls = []

    def post(self, url, json=None, headers=None, timeout=None):
        self.calls.append({"url": url, "json": json, "headers": headers or {}})
        return self.responses.pop(0)


session = FakeSession(
    [
        FakeResponse(402, {"x402Version": 1, "accepts": [{"scheme": "exact", "network": "base-sepolia"}]}),
    ]
)
client = x402_secure.X402SecureClient(base_url="https://x402-secure.example", session=session)
first_result = client.score_endpoint("https://merchant.example/x402")
# On a 402 the tool returns the plugin's "PAYMENT_REQUIRED: " marker string (not a dict),
# so AgentCorePaymentsPlugin can detect it and retry with an x402 payment header.
assert x402_secure.is_payment_required(first_result)

retry_session = FakeSession(
    [
        FakeResponse(200, {"overall_score": 91, "risk_level": "low", "is_scam": False}),
    ]
)
retry_client = x402_secure.X402SecureClient(base_url="https://x402-secure.example", session=retry_session)
score = retry_client.score_endpoint(
    "https://merchant.example/x402",
    headers={"X-PAYMENT": "signed"},
)
assert score["overall_score"] == 91
assert retry_session.calls[0]["headers"]["X-PAYMENT"] == "signed"
score

## AgentCore Runtime Deploy, Invoke, Observe

After local verification and IAM review, deploy the runtime with the **AgentCore CLI** (`@aws/agentcore`, CDK-based). These are operator steps, not notebook-executed cells, because they create cloud resources and can trigger live x402 payment flows when invoked with funded payment context. (The Python `bedrock-agentcore-starter-toolkit` with `configure`/`launch` is deprecated — don't use it.)

Install the current AgentCore Payments beta service model into `~/.aws/models` first (see `setup/README.md`), then prepare payment resources and deploy:

```bash
# Payment resources
bash setup/setup_roles.sh
bash setup/setup_manager.sh
bash setup/setup_instrument.sh   # EMBEDDED_CRYPTO_WALLET instrument + payment session

# Deploy with the AgentCore CLI (requires Node.js 20+ and AWS credentials)
npm install -g @aws/agentcore
agentcore create                 # scaffold a project; add this agent as BYO (agentcore add agent --type byo)
agentcore deploy                 # build + deploy to AgentCore Runtime via CDK
```

Before invoking, complete the one-time onboarding the instrument step prints: a real `INSTRUMENT_EMAIL`, **both** delegated-signing layers (project policy + per-wallet WalletHub grant), and fund the wallet with USDC on Base mainnet. Then invoke with `agentcore invoke` (or from your application backend) with a prompt and per-invocation `payment_context`. The expected observability shape (`agentcore logs` / `agentcore traces`, or CloudWatch GenAI Observability) is an agent turn that calls `check_x402_endpoint_trust`, lets `AgentCorePaymentsPlugin` process the t54 x402-secure HTTP 402 retry, then calls `call_trusted_x402_service` only after the deterministic guardrail passes.

For a local run instead of a cloud deploy: `agentcore dev` (hot-reload), or run the agent directly with `PYTHONPATH="$PWD/agent" python agent/agent.py "<prompt>"`.

When finished, tear down the runtime resources (stops billing):

```bash
agentcore remove
```

See the [AgentCore CLI docs](https://github.com/aws/agentcore-cli) for the full project-setup wizard and BYO-agent options.

## Optional AWS/x402 Gates

The shell and Python scripts under `test/integration/` are safety gates. They confirm account configuration and required variables before a live test. They do not execute a live paid data request by default.

In [ ]:
if os.environ.get("RUN_AWS_X402_E2E") != "1":
    print("AWS/x402 prerequisite gates: SKIPPED")
else:
    print("Run the integration scripts from a terminal after reviewing .env and funding limits.")

## Non-Goals

- This notebook does not create AWS resources automatically. Use `setup/` scripts after reviewing IAM scope.
- This notebook does not run a live t54 x402-secure payment or target data payment by default.
- This sample uses t54's direct API integration, not the x402-secure SDK wrapper or facilitator proxy mode.
- The production path relies on `AgentCorePaymentsPlugin` for HTTP 402 payment retry; the lower-level manual x402 helpers are retained only for compatibility tests.